In [ ]:
## In this example, we will study output data frame from pandora.py configuration
#### 1. Opening each data frame and check structure
#### 2. Collect POT and scale factor to the target POT
#### 3. Merge evtdf and mcnudf for further study
#### 4. Draw some plots for each slice and for each pfp

import os
import sys
print('Importing cafpyana utils...')
cafpyana_root = "/home/lpelegri/cafpyana"
# Add CAFpyana to the Python search path -- this will allow you to import cafpyana modules
sys.path.insert(0, cafpyana_root)

import lmfit
import numpy as np
import math
import uproot as uproot
import pickle
import pandas as pd

print("Loading")
import matplotlib.pyplot as plt
import matplotlib.colors
from matplotlib.colors import LinearSegmentedColormap
from matplotlib import ticker
from matplotlib.ticker import (AutoMinorLocator, MultipleLocator)
from matplotlib import gridspec

from analysis_village.unfolding.wienersvd import *
from analysis_village.unfolding.unfolding_inputs import *

# Add the head direcoty to sys.path
workspace_root = os.getcwd()  
sys.path.insert(0, workspace_root + "/../../")


from analysis_village.cc1pi.var_configs import *

from analysis_village.cc1pi.HelperFunctions import HelperFunctions
from analysis_village.cc1pi.Constants import CTE as CTE
from analysis_village.cc1pi.CutMasks.CutMasks import *
from analysis_village.cc1pi.CutMasks.MaskUtils import *
from analysis_village.cc1pi.DataFrameUtils import DFCleaning
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from analysis_village.cc1pi.GraphUtils.GraphsUtils import *
from analysis_village.cc1pi.GraphUtils.Utils import *

# import this repo's classes
import pyanalib.pandas_helpers as ph
import pyanalib.split_df_helpers as splh
import pyanalib.stat_helpers as sh

from pyanalib.split_df_helpers import *
from analysis_village.cc1pi.systematics.final_variable_configs import VariableConfig
from analysis_village.cc1pi.systematics.utils import *
from analysis_village.cc1pi.systematics.constants import *
from pyanalib.covariance import *
from analysis_village.cc1pi.DataFrameUtils.DFLoading import *
from cols_to_keep import *

np.seterr(divide='ignore', invalid='ignore', over='ignore')

In [ ]:
#Load CV dataframe
keys2load = ["pfp", "hdr", "histpotdf","hit0","hit1","hit2"] ## keys from the configuration file
#bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_muon.df"
bnb_path = "/exp/sbnd/data/users/lpelegri/TLECafpyanaData/TLE_1e20_pion_stopping.df"
mc_bnb_df = load_df(bnb_path, keys2load, 100)
mc_bnb_pfp_df = mc_bnb_df['pfp']
mc_bnb_hit0_df = mc_bnb_df['hit0']
mc_bnb_hit1_df = mc_bnb_df['hit1']
mc_bnb_hit2_df = mc_bnb_df['hit2']
mc_bnb_hdr_df = mc_bnb_df['hdr']

In [ ]:
mc_bnb_hit0_df.columns

In [ ]:
pot_weight_col = ('slc', 'wgt', '', '', '', '')

print("data_tot_pot: %.3e" %(data_tot_pot))
mc_tot_pot = mc_bnb_hdr_df['pot'].sum()
print("mc_tot_pot: %.3e" %(mc_tot_pot))
mc_pot_scale = data_tot_pot / mc_tot_pot
print("mc_pot_scale: %.3e" %(mc_pot_scale))
mc_bnb_pfp_df[pot_weight_col] = mc_pot_scale * np.ones(len(mc_bnb_pfp_df))

In [ ]:
import pandas as pd

# Define the target 4 index levels identifying unique tracks/slices
target_levels = ['__ntuple', 'entry', 'rec.slc..index', 'rec.slc.reco.pfp..index']

# --- 1. Combined Unique Combinations Across hit0 + hit1 + hit2 ---
hit_dfs = {
    '0': mc_bnb_hit0_df,
    '1': mc_bnb_hit1_df,
    '2': mc_bnb_hit2_df,
}

hit_tuple_sets = []
total_hit_rows = 0

for name, df in hit_dfs.items():
    if not df.empty:
        total_hit_rows += len(df)
        # Extract target 4-tuples as a set
        tuples_set = set(
            df.index.to_frame()[target_levels].itertuples(index=False, name=None)
        )
        hit_tuple_sets.append(tuples_set)

# Union of unique 4-tuples across hit0, hit1, and hit2
if hit_tuple_sets:
    combined_hit_unique_tuples = set.union(*hit_tuple_sets)
    n_unique_combined_hits = len(combined_hit_unique_tuples)
else:
    n_unique_combined_hits = 0

print(f"Combined (hit0+hit1+hit2) total hit rows: {total_hit_rows}")
print(f"Combined (hit0+hit1+hit2) unique 4-tuple combinations: {n_unique_combined_hits}")

print("\n" + "=" * 50 + "\n")

# --- 2. Unique Combinations for mc_bnb_pfp_df ---
mc_bnb_pfp_df = mc_bnb_pfp_df.sort_index(level='__ntuple', ascending=True)

pfp_tuples_set = set(
    mc_bnb_pfp_df.index.to_frame()[target_levels].itertuples(index=False, name=None)
)
n_unique_pfp = len(pfp_tuples_set)

print(f"mc_bnb_pfp_df total rows: {len(mc_bnb_pfp_df)}")
print(f"mc_bnb_pfp_df unique 4-tuple combinations: {n_unique_pfp}")

# --- Optional Check: Overlap between Hits and PFP ---
if n_unique_combined_hits > 0 and n_unique_pfp > 0:
    overlap = len(combined_hit_unique_tuples.intersection(pfp_tuples_set))
    print("\n" + "=" * 50 + "\n")
    print(f"Unique tracks present in BOTH PFP and Hits: {overlap}")

In [ ]:
import pandas as pd

# Define your columns
p_type_col = ('pfp', 'trk', 'truth', 'p', 'p_type', '')
weight_col = ('slc', 'wgt', '', '', '', '')  # Optional: set to None if unweighted

# --- 1. Extract and Clean Data ---
valid_mask = mc_bnb_pfp_df[p_type_col].notna()
df_clean = mc_bnb_pfp_df[valid_mask]

if weight_col and weight_col in df_clean.columns:
    weights = df_clean[weight_col].fillna(1.0)
else:
    weights = pd.Series(1.0, index=df_clean.index)

# --- 2. Calculate Weighted & Unweighted Statistics ---
stats_df = pd.DataFrame({
    'p_type': df_clean[p_type_col],
    'weight': weights
})

summary = stats_df.groupby('p_type').agg(
    Counts=('weight', 'count'),
    Weighted_Yield=('weight', 'sum')
).reset_index()

total_counts = summary['Counts'].sum()
total_weighted = summary['Weighted_Yield'].sum()

summary['Raw_%'] = (summary['Counts'] / total_counts) * 100
summary['Weighted_%'] = (summary['Weighted_Yield'] / total_weighted) * 100

# Sort by weighted yield (descending)
summary = summary.sort_values(by='Weighted_Yield', ascending=False)

# --- 3. Pretty Print Output ---
print("\n" + "="*60)
print(f"{'p_type':<15} | {'Counts':<8} | {'Raw %':<8} | {'Weighted':<10} | {'Weighted %':<10}")
print("-" * 60)

for _, row in summary.iterrows():
    print(f"{str(row['p_type']):<15} | {int(row['Counts']):<8d} | {row['Raw_%']:<7.2f}% | {row['Weighted_Yield']:<10.1f} | {row['Weighted_%']:<9.2f}%")

print("-" * 60)
print(f"{'Total':<15} | {total_counts:<8d} | {100.0:<7.2f}% | {total_weighted:<10.1f} | {100.0:<9.2f}%")
print("="*60 + "\n")

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# Default bin definitions
DEFAULT_BINS_Y = np.linspace(0, 10, 51)  # dE/dx range [MeV/cm]
DEFAULT_BINS_Z = np.linspace(0, 200, 51) # Residual range [cm]


def plot_split_tpc_2d(
    df: pd.DataFrame,
    x_col: str = "rr",
    y_col: str = "dedx",
    x_split_col: str = "x",
    weight_col: str = None,
    bins_x: np.ndarray = DEFAULT_BINS_Z,
    bins_y: np.ndarray = DEFAULT_BINS_Y,
    xlabel: str = "Residual Range [cm]",
    ylabel: str = "dE/dx [MeV/cm]",
    title_prefix: str = "Hit Distribution",
    cmap_name: str = "viridis",
    figsize: tuple = (15, 6),
):
    """Generates a 1x2 2D histogram multiplot split by X < 0 (left) and X >= 0 (right)."""

    # Clean data & extract arrays safely
    mask = df[x_col].notna() & df[y_col].notna() & df[x_split_col].notna()
    plot_df = df[mask]

    x_vals = plot_df[x_col].values
    y_vals = plot_df[y_col].values
    split_vals = plot_df[x_split_col].values

    if weight_col and weight_col in plot_df.columns:
        weights = plot_df[weight_col].fillna(1.0).values
    else:
        weights = np.ones_like(x_vals)

    finite_mask = (
        np.isfinite(x_vals)
        & np.isfinite(y_vals)
        & np.isfinite(split_vals)
        & np.isfinite(weights)
    )
    x_vals, y_vals, split_vals, weights = (
        x_vals[finite_mask],
        y_vals[finite_mask],
        split_vals[finite_mask],
        weights[finite_mask],
    )

    # Subdivide by TPC side using x_split_col
    mask_neg_x = split_vals < 0
    mask_pos_x = split_vals >= 0

    # Setup 1x2 Subplots with shared Y-axis
    fig, (ax_left, ax_right) = plt.subplots(
        1, 2, figsize=figsize, sharey=True, gridspec_kw={"wspace": 0.08}
    )

    cmap = globals().get("sunset_cmap", cmap_name)

    # Calculate global max for uniform colorbar scaling
    h_left, _, _ = np.histogram2d(
        x_vals[mask_neg_x], y_vals[mask_neg_x], bins=[bins_x, bins_y], weights=weights[mask_neg_x]
    )
    h_right, _, _ = np.histogram2d(
        x_vals[mask_pos_x], y_vals[mask_pos_x], bins=[bins_x, bins_y], weights=weights[mask_pos_x]
    )
    vmax = max(h_left.max(), h_right.max())
    vmax = vmax if vmax > 0 else None

    # --- Left Plot: Split Var < 0 ---
    im0 = ax_left.hist2d(
        x_vals[mask_neg_x],
        y_vals[mask_neg_x],
        bins=[bins_x, bins_y],
        weights=weights[mask_neg_x],
        cmap=cmap,
        cmin=1e-5,
        vmax=vmax,
    )[3]

    ax_left.set_title(f"{title_prefix}: $X < 0$ cm", fontsize=14, pad=10)
    ax_left.set_xlabel(xlabel, fontsize=14)
    ax_left.set_ylabel(ylabel, fontsize=14)
    ax_left.set_xlim(bins_x[0], bins_x[-1])
    ax_left.set_ylim(bins_y[0], bins_y[-1])
    ax_left.grid(alpha=0.3, linestyle="--")

    # --- Right Plot: Split Var >= 0 ---
    im1 = ax_right.hist2d(
        x_vals[mask_pos_x],
        y_vals[mask_pos_x],
        bins=[bins_x, bins_y],
        weights=weights[mask_pos_x],
        cmap=cmap,
        cmin=1e-5,
        vmax=vmax,
    )[3]

    ax_right.set_title(f"{title_prefix}: $X \\geq 0$ cm", fontsize=14, pad=10)
    ax_right.set_xlabel(xlabel, fontsize=14)
    ax_right.set_xlim(bins_x[0], bins_x[-1])
    ax_right.grid(alpha=0.3, linestyle="--")

    # Common Colorbar
    cbar = fig.colorbar(im1, ax=[ax_left, ax_right], pad=0.02)
    cbar.set_label("Weighted Entries", fontsize=12)

    return fig, (ax_left, ax_right)

In [ ]:
# Pass plain string column names
for name, hitdf in hit_dfs.items():
    fig, axes = plot_split_tpc_2d(
        df=hitdf,
        x_col="rr",
        y_col="dedx",
        x_split_col="x",
        weight_col=None,
        bins_x=np.linspace(0, 80, 41),   # Residual Range [cm]
        bins_y=np.linspace(0, 10, 41),    # dE/dx [MeV/cm]
        xlabel="Residual Range [cm]",
        ylabel="Hit dE/dx [MeV/cm]",
        title_prefix=f"Plane {name} dE/dx vs RR",
    )

plt.show()

In [ ]:
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


def plot_dedx_by_rr_and_tpc(
    df: pd.DataFrame,
    rr_range: tuple = (0.0, 1.0),
    dedx_col: str = "dedx",
    rr_col: str = "rr",
    x_col: str = "x",
    weight_col: str = None,
    bins: np.ndarray = np.linspace(0.0, 10.0, 51),
    stacked: bool = False,
    density: bool = False,  # 👈 Added parameter
    alpha: float = 0.3,
    linewidth: float = 1.8,
    figsize: tuple = (8, 6),
    title: str = None,
    ax: plt.Axes = None,
):
    # --- 1. Filter RR Range & Clean Data ---
    mask = (
        df[dedx_col].notna()
        & df[rr_col].notna()
        & df[x_col].notna()
        & (df[rr_col] >= rr_range[0])
        & (df[rr_col] < rr_range[1])
    )
    plot_df = df[mask].copy()

    if plot_df.empty:
        raise ValueError(
            f"No valid entries found for {rr_col} in range {rr_range}."
        )

    # Resolve Weights
    if weight_col is not None and weight_col in plot_df.columns:
        weights = plot_df[weight_col].fillna(1.0)
    else:
        weights = pd.Series(1.0, index=plot_df.index)

    # --- 2. Categorization by X Position ---
    mask_pos_x = plot_df[x_col] > 0
    mask_neg_x = ~mask_pos_x

    grouped_data = [
        plot_df.loc[mask_pos_x, dedx_col],
        plot_df.loc[mask_neg_x, dedx_col],
    ]
    grouped_weights = [
        weights[mask_pos_x],
        weights[mask_neg_x],
    ]

    # --- Handle Normalization (density=True) ---
    bin_width = bins[1] - bins[0]
    if density:
        if stacked:
            # Normalize so the combined stacked area sums to 1.0
            total_weight = sum(w.sum() for w in grouped_weights)
            if total_weight > 0:
                scale = 1.0 / (total_weight * bin_width)
                grouped_weights = [w * scale for w in grouped_weights]
        else:
            # Normalize each subgroup independently so each group's area sums to 1.0
            normed_weights = []
            for w in grouped_weights:
                w_sum = w.sum()
                if w_sum > 0:
                    normed_weights.append(w / (w_sum * bin_width))
                else:
                    normed_weights.append(w)
            grouped_weights = normed_weights

    colors = ["#1f77b4", "#d62728"]  # Blue & Red
    labels = [r"$X > 0$ cm", r"$X \leq 0$ cm"]

    # --- 3. Plotting ---
    if ax is None:
        fig, ax = plt.subplots(figsize=figsize)
    else:
        fig = ax.get_figure()

    # Layer 1: Filled steps
    ax.hist(
        grouped_data,
        bins=bins,
        weights=grouped_weights,
        stacked=stacked,
        histtype="stepfilled",
        color=colors,
        alpha=alpha,
        label=labels,
    )

    # Layer 2: Outlines
    ax.hist(
        grouped_data,
        bins=bins,
        weights=grouped_weights,
        stacked=stacked,
        histtype="step",
        color=colors,
        linewidth=linewidth,
    )

    # --- 4. Styling & Formatting ---
    ax.set_xlabel(r"Hit $dE/dx$ [MeV/cm]", fontsize=14)

    if density:
        ax.set_ylabel("A.U.", fontsize=14)
    elif weight_col:
        ax.set_ylabel("Weighted Hits", fontsize=14)
    else:
        ax.set_ylabel("Hits", fontsize=14)

    if title is None:
        title = rf"Hit $dE/dx$ Distribution ({rr_range[0]} $\leq$ RR < {rr_range[1]} cm)"
    ax.set_title(title, fontsize=15, pad=12)

    ax.set_xlim(bins[0], bins[-1])
    ax.set_ylim(0, ax.get_ylim()[1] * 1.15)

    ax.tick_params(axis="both", which="both", labelsize=12, direction="in")
    ax.grid(axis="y", linestyle="--", alpha=0.4, zorder=0)

    ax.legend(
        fontsize=12,
        frameon=True,
        framealpha=1.0,
        edgecolor="black",
        fancybox=False,
    )

    return fig, ax

In [ ]:
hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
rr_ranges = [(2, 3), (6, 7), (15, 16)]

for rr_min, rr_max in rr_ranges:
    fig, axes = plt.subplots(
        1, len(hit_dfs), figsize=(18, 5), sharey=True, gridspec_kw={"wspace": 0.08}
    )

    max_y_value = 0  # Track global maximum density across planes

    for plane_idx, (df_plane, ax) in enumerate(zip(hit_dfs, axes)):
        try:
            plot_dedx_by_rr_and_tpc(
                df=df_plane,
                rr_range=(rr_min, rr_max),
                dedx_col="dedx",
                rr_col="rr",
                x_col="x",
                weight_col=None,
                bins=np.linspace(0.0, 10.0, 51),
                stacked=False,
                density=True,  # 👈 Now supported!
                title=f"Plane {plane_idx}",
                ax=ax,
            )

            # Record maximum bin height in this subplot
            current_max = ax.get_ylim()[1] / 1.15
            if current_max > max_y_value:
                max_y_value = current_max

        except ValueError:
            ax.set_title(f"Plane {plane_idx}: No Data", fontsize=15, pad=12)
            continue

        # Clean up side subplots
        if plane_idx > 0:
            ax.set_ylabel("")
            legend = ax.get_legend()
            if legend:
                legend.remove()

    # Apply the global max limit + 15% headroom to all subplots
    if max_y_value > 0:
        axes[0].set_ylim(0, max_y_value * 1.15)

    fig.suptitle(
        rf"Normalized Hit $dE/dx$ Comparison ({rr_min} $\leq$ RR < {rr_max} cm)",
        fontsize=16,
        y=1.03,
    )

    plt.show()

In [ ]:
"""
langau_rr_fit.py
=================
Python/PyROOT port of the ROOT macro `FitPlotWholeDataset.C`.

Instead of building TH2D(rr, dEdx) histograms from a ROOT file and slicing
them bin-by-bin, this version works directly on per-hit pandas dataframes
(e.g. hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df], one
dataframe per wire plane, each carrying a 'tpc' column for TPC0/TPC1).
For each (plane, tpc) pair the hits are sliced in residual range (rr),
and the dE/dx distribution of each slice is fit with the same two-stage
Landau-convoluted-with-Gaussian ("Langau") fit as the original macro.
The resulting Gaussian-smearing sigma is then fit vs. MPV with the same
power law used by `f_pol2` in the macro.

Requirements: numpy, pandas, matplotlib, scipy, and a PyROOT build of ROOT
(needed for TH1D/TF1 and TMath::Landau/Gaus, which is the actual fitting
engine -- reimplementing the Landau/Gaussian convolution fit in pure numpy
would give a different, unvalidated minimizer).

-----------------------------------------------------------------------
Mapping from the original macro to this module
-----------------------------------------------------------------------
FitPlotWholeDataset.C construct              -> here
--------------------------------------------  ------------------------------
langaufun                                     -> langau_cpp (declared via
                                                  ROOT.gInterpreter.Declare)
langaufit                                     -> _langau_fit_once
broad fit, then MPV-centered refit            -> langau_fit_two_stage
TH2D "slice_xbin_%d" loop over ix             -> fit_rr_slices (slices the
                                                  hit-level dataframe by rr
                                                  directly, no TH2D needed)
gr_MPV_gaussian_smearing[i] + f_pol2[i] fit   -> per-(plane, tpc) DataFrame
                                                  + fit_sigma_vs_mpv
canvas c3 (all 6 planes/tpc overlay)          -> plot_all_planes
canvas c4 (3 subplots, tpc0 vs tpc1)          -> plot_by_plane
PhysdEdx / Hypfit "theoretical" MPV           -> load_physics_classes +
                                                  make_theoretical_mpv_func
                                                  (see note below)

Note on the "theoretical" MPV
------------------------------
The original macro plots sigma_G against a *theoretical* MPV coming from
`hfit.map_PhysdEdx[PDG]` (Bethe-Bloch mean dE/dx, Landau's xi, Wmax, and a
KE-from-range spline) rather than the fitted MPV of the slice. Since
Hypfit.h/.cpp and PhysdEdx.h/.cpp are available next to this script, this
module loads the *real* classes straight into the ROOT interpreter (the
same way the macro did with `#include "PhysdEdx.h"`/`"PhysdEdx.cpp"`) via
`load_physics_classes`, then reproduces the macro's
KEFromRangeSpline/Landau_xi/Get_Wmax/meandEdx/dEdx_PDF_fuction block in
`make_theoretical_mpv_func`, rather than approximating the physics.
If those files aren't available in a given run, `theoretical_mpv_func`
can simply be left as `None` and the *fitted* MPV of each slice is used
instead (mirrors the macro's commented-out "Doing it with reco" branch).
"""

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.optimize import curve_fit

import ROOT
from ROOT import TH1D, TF1

# ===========================================================================
# 1. Landau (x) Gaussian convolution -- ported 1:1 from `langaufun`
#    Parameter order matches the original macro:
#       p[0] = Width   (Landau scale)
#       p[1] = MPV     (Landau most probable value)
#       p[2] = Area
#       p[3] = GSigma  (Gaussian smearing sigma)
# ===========================================================================
ROOT.gInterpreter.Declare(r"""
#include "TMath.h"

double langau_cpp(double *x, double *p) {
    const double width = p[0];
    const double mpv   = p[1];
    const double area  = p[2];
    const double sigG  = p[3];

    if (width <= 0 || sigG <= 0) return 0.0;

    const double invsq2pi = 0.398942280401;
    const double np = 500.0;   // integration slices, same as the macro
    const double sc = 5.0;     // convolution extends +/- sc * sigG
    const double xx = x[0];

    double xlow = xx - sc * sigG;
    double xupp = xx + sc * sigG;
    double step = (xupp - xlow) / np;

    double sum = 0.0;
    for (double i = 1.0; i <= np / 2.0; i += 1.0) {
        double t1 = xlow + (i - 0.5) * step;
        double fland1 = TMath::Landau(t1, mpv, width) / width;
        sum += fland1 * TMath::Gaus(xx, t1, sigG);

        double t2 = xupp - (i - 0.5) * step;
        double fland2 = TMath::Landau(t2, mpv, width) / width;
        sum += fland2 * TMath::Gaus(xx, t2, sigG);
    }
    return area * step * sum * invsq2pi / sigG;
}
""")


import os
import ROOT


def load_physics_classes():
    base_dir = (
        "/home/lpelegri/cafpyana/analysis_village/cc1pi/TLExtensionMethod"
    )

    # Compile and load libraries dynamically
    res1 = ROOT.gSystem.CompileMacro(
        os.path.join(base_dir, "PhysdEdx.cpp"), "k"
    )
    res2 = ROOT.gSystem.CompileMacro(os.path.join(base_dir, "Hypfit.cpp"), "k")

    if res1 != 1 or res2 != 1:
        raise RuntimeError(
            "C++ compilation failed! Check stdout for syntax or redefinition errors."
        )

    return ROOT.Hypfit()

In [ ]:
hfit = load_physics_classes()

In [ ]:


# ===========================================================================
# 2. Histogram construction from a pandas Series (ported from
#    `th1_from_series`, simplified to fixed binning / no weights, since the
#    hit dataframes don't carry a weight column)
# ===========================================================================
def th1_from_series(values, name, title="", nbins=100, xmin=0.0, xmax=20.0):
    """Build a TH1D (with Sumw2) from a 1D array-like of dE/dx values."""
    vals = np.asarray(values, dtype=float)
    vals = vals[np.isfinite(vals)]
    counts, _edges = np.histogram(vals, bins=nbins, range=(xmin, xmax))
    h = TH1D(name, title, nbins, xmin, xmax)
    h.Sumw2()
    for i, c in enumerate(counts, start=1):
        h.SetBinContent(i, float(c))
        h.SetBinError(i, np.sqrt(c) if c > 0 else 1.0)
    return h


# ===========================================================================
# 3. Two-stage Langau fit, ported from `langaufit` + the broad-then-refined
#    double-fit logic that lived inline in FitPlotWholeDataset()
# ===========================================================================
def _langau_fit_once(hist, frange, start, lo, hi, fname):
    old = ROOT.gROOT.GetListOfFunctions().FindObject(fname)
    if old:
        old.Delete()

    f = TF1(fname, ROOT.langau_cpp, frange[0], frange[1], 4)
    f.SetParameters(*start)
    f.SetParNames("Width", "MPV", "Area", "GSigma")
    for i in range(4):
        f.SetParLimits(i, lo[i], hi[i])

    fitres = hist.Fit(fname, "RBOSQN")  # range, bounded, no-draw, store, quiet, no-store-in-hist
    pars = [f.GetParameter(i) for i in range(4)]
    errs = [f.GetParError(i) for i in range(4)]
    chi2 = f.GetChisquare()
    ndf = f.GetNDF()
    status = fitres.CovMatrixStatus()
    return f, pars, errs, chi2, ndf, status


def langau_fit_two_stage(hist, first_stage_range=(0.0, 20.0), max_par_err=1.0):
    """
    Stage 1: broad fit over `first_stage_range` to locate the peak (mirrors
             the first `langaufit(...)` call in the macro).
    Stage 2: refit restricted to [0.8*MPV, 1.5*MPV], seeded with the stage-1
             parameters (mirrors the macro's second `langaufit(...)` call).
    Slices whose stage-2 MPV or GSigma error exceeds `max_par_err` are
    rejected, exactly like the `if (perrs[3] > 1) continue;` /
    `if (perrs[1] > 1) continue;` guards in the macro.

    Returns a dict(func, pars, errs, chi2, ndf, status), or None if the
    slice should be dropped.
    """
    max_x = hist.GetBinCenter(hist.GetMaximumBin())
    bin_width = hist.GetBinWidth(1)

    sv = [0.1, max_x, hist.Integral() * 0.05 * bin_width, 0.2]
    lo = [0.01 * v if v != 0 else 0.0 for v in sv]
    hi = [100.0 * v if v != 0 else 1.0 for v in sv]

    try:
        _f1, p1, _e1, _c1, _n1, _s1 = _langau_fit_once(
            hist, first_stage_range, sv, lo, hi, "langau_stage1"
        )
    except Exception:
        return None

    mpv1 = p1[1]
    if not np.isfinite(mpv1) or mpv1 <= 0:
        return None
    second_range = (mpv1 * 0.8, mpv1 * 1.5)

    try:
        f2, p2, e2, c2, n2, s2 = _langau_fit_once(
            hist, second_range, p1, lo, hi, "langau_stage2"
        )
    except Exception:
        return None

    if e2[3] > max_par_err or e2[1] > max_par_err:
        return None

    return dict(func=f2, pars=p2, errs=e2, chi2=c2, ndf=n2, status=s2)


# ===========================================================================
# 3b. Load the real PhysdEdx / Hypfit classes and build the theoretical-MPV
#     function from them, mirroring the macro's
#         Hypfit hfit;
#         ... hfit.map_PhysdEdx[PDG]->KEFromRangeSpline/Landau_xi/Get_Wmax/
#             meandEdx ...
#         TF1 *PDF = new TF1("", dEdx_PDF_fuction, -10., 20., 5);
#     block. Only needed if you want the theoretical MPV instead of the
#     default reco/fitted one -- safe to skip if you just want the reco
#     version from section 4/5 below.
# ===========================================================================
PDG_MASS = {
    13: 105.6583755, -13: 105.6583755,      # muon
    211: 139.57039, -211: 139.57039,        # charged pion
    2212: 938.27208943,                     # proton
}




def build_theoretical_pdf(hfit, pdg, rr_center, mean_pitch, mass=None):
    """
    Reproduces the macro's block:

        this_KE      = hfit.map_PhysdEdx[PDG]->KEFromRangeSpline(rr_bin)
        gamma        = this_KE / mass + 1.0
        beta2        = 1 - 1/gamma^2
        this_xi      = hfit.map_PhysdEdx[PDG]->Landau_xi(this_KE, pitch)
        this_Wmax    = hfit.map_PhysdEdx[PDG]->Get_Wmax(this_KE)
        this_kappa   = this_xi / this_Wmax
        this_dEdx_BB = hfit.map_PhysdEdx[PDG]->meandEdx(this_KE)
        TF1 *PDF = new TF1("", dEdx_PDF_fuction, -10., 20., 5);
        PDF->SetParameters(kappa, beta2, xi, dEdx_BB, pitch);

    and returns the *configured TF1 itself*, so callers can use
    `.GetMaximumX()` for the MPV or `.Eval(x)` to overlay the curve on a plot.

    IMPORTANT: the 2nd argument to TF1 here is the actual function object
    `ROOT.PhysdEdx.dEdx_PDF_function`, NOT the string `"dEdx_PDF_function"`.
    `TF1(name, "some string", xmin, xmax, npar)` is a *different*
    constructor overload -- it treats the string as a TFormula expression to
    compile (like `"[0]+[1]*x"`), not as "the name of a C++ function to
    call". That's exactly what produced the
    `Error in <TFormula::ProcessFormula>: "dEdx_PDF_function" has not been
    matched in the formula expression` error: Cling tried to compile
    `dEdx_PDF_function` as a bare formula token and failed (a stray global
    left over from an earlier ROOT session with a TF1 literally named
    "dEdx_PDF_function" can even make that error look like it half-resolved
    -- ROOT auto-creates a global pointer under a TNamed's own name, so
    reusing that exact string as a TF1 name anywhere is worth avoiding).
    Passing the callable instead avoids all of this.
    """
    if mass is None:
        mass = PDG_MASS.get(abs(pdg), 105.6583755)

    phys = hfit.map_PhysdEdx[pdg]
    this_KE = phys.KEFromRangeSpline(rr_center)
    gamma = (this_KE / mass) + 1.0
    beta2 = 1.0 - 1.0 / (gamma * gamma)
    this_xi = phys.Landau_xi(this_KE, mean_pitch)
    this_Wmax = phys.Get_Wmax(this_KE)
    this_kappa = this_xi / this_Wmax
    this_dEdx_BB = phys.meandEdx(this_KE)

    pdf = ROOT.TF1("", ROOT.PhysdEdx.dEdx_PDF_function, -10.0, 20.0, 5)
    pdf.SetParameters(this_kappa, beta2, this_xi, this_dEdx_BB, mean_pitch)
    return pdf


def make_theoretical_mpv_func(hfit, pdg, mass=None):
    """
    Thin wrapper around `build_theoretical_pdf` for `fit_rr_slices`/
    `analyze`, which only need the scalar MPV (`PDF.GetMaximumX()`), not
    the full TF1. `hfit` is whatever `load_physics_classes()` returned;
    `pdg` is the PDG code (13=mu-, 211=pi+, 2212=proton, ...); `mass`
    defaults to the matching entry in `PDG_MASS` (MeV) if not given.
    """
    def theoretical_mpv(rr_center, mean_pitch):
        pdf = build_theoretical_pdf(hfit, pdg, rr_center, mean_pitch, mass=mass)
        return pdf.GetMaximumX()
    return theoretical_mpv


# ===========================================================================
# 4. Slice a per-hit dataframe in residual range and fit each slice.
#    This replaces the TH2D "slice_xbin_%d" loop over `ix` in the macro,
#    since we now have per-hit rr/dedx values directly instead of a
#    pre-filled 2D histogram.
# ===========================================================================
def make_rr_edges(rr_min, rr_max, bin_width):
    return np.arange(rr_min, rr_max + bin_width, bin_width)


def fit_rr_slices(df, plane, tpc, dedx_col="dedx",
                   rr_min=3.0, rr_max=40.0, rr_bin_width=1.0,
                   hist_nbins=100, hist_xmin=0.0, hist_xmax=20.0,
                   min_entries=30, first_stage_range=(0.0, 20.0),
                   theoretical_mpv_func=None, verbose=False):
    """
    x-axis of the returned graph (column 'mpv_x'):
      - if `theoretical_mpv_func(rr_center, mean_pitch)` is supplied, its
        return value is used (mirrors `PDF->GetMaximumX()` built from
        PhysdEdx's Bethe-Bloch/Landau theory in the macro).
      - otherwise the *fitted* MPV of the slice is used (mirrors the
        macro's commented-out "Doing it with reco" branch).
    """
    edges = make_rr_edges(rr_min, rr_max, rr_bin_width)
    rows = []
    for lo, hi in zip(edges[:-1], edges[1:]):
        sl = df[(df["rr"] >= lo) & (df["rr"] < hi)]
        if len(sl) < min_entries:
            continue

        rr_center = 0.5 * (lo + hi)
        hname = f"slice_p{plane}_t{tpc}_rr{rr_center:.2f}"
        hist = th1_from_series(sl[dedx_col], hname, "", hist_nbins, hist_xmin, hist_xmax)

        fit = langau_fit_two_stage(hist, first_stage_range=first_stage_range)
        if fit is None:
            if verbose:
                print(f"  [plane {plane}, tpc {tpc}] rr={rr_center:.2f}: fit rejected")
            continue

        width, mpv_reco, area, gsigma = fit["pars"]
        w_e, mpv_e, area_e, gsigma_e = fit["errs"]

        if theoretical_mpv_func is not None:
            mean_pitch = sl["pitch"].mean()
            mpv_x = theoretical_mpv_func(rr_center, mean_pitch)
            mpv_x_err = 0.0
        else:
            mpv_x = mpv_reco
            mpv_x_err = mpv_e

        rows.append(dict(
            plane=plane, tpc=tpc, rr_center=rr_center, n_hits=len(sl),
            mpv_x=mpv_x, mpv_x_err=mpv_x_err,
            mpv_reco=mpv_reco, mpv_reco_err=mpv_e,
            width=width, width_err=w_e,
            area=area, area_err=area_e,
            gsigma=gsigma, gsigma_err=gsigma_e,
            chi2=fit["chi2"], ndf=fit["ndf"], status=fit["status"],
        ))

    return pd.DataFrame(rows)


# ===========================================================================
# 5. sigma_G(MPV) power-law fit -- ported from the `f_pol2` TF1 fit
#    ("[0] + [1]*x^[2]") on gr_MPV_gaussian_smearing[i]
# ===========================================================================
def power_law(x, a, b, c):
    return a + b * np.power(x, c)


def fit_sigma_vs_mpv(res_df, x_col="mpv_x", y_col="gsigma", yerr_col="gsigma_err",
                      p0=(0.2, 0.1, 1.5),
                      bounds=((0.0, 0.0, -10.0), (1e6, 1.0, 10.0))):
    """
    NOTE: in the macro, `f_pol2[i]->SetParLimits(0, 0, 1e6)` and
    `SetParLimits(1, 0, 1)` are applied to parameters 0 and 1, but the
    adjacent comments describe constraining "[2] > 0" and "[0] < 1" --
    i.e. the comments and the code don't line up. The bounds below
    constrain a>=0 and 0<=b<=1 (matching the code, not the comments);
    change them if you intended the commented behavior instead.
    """
    x = res_df[x_col].to_numpy()
    y = res_df[y_col].to_numpy()
    yerr = res_df[yerr_col].to_numpy()
    yerr = np.where(yerr > 0, yerr, np.nan)
    fallback = np.nanmedian(yerr)
    yerr = np.where(np.isfinite(yerr), yerr, fallback if np.isfinite(fallback) else 1.0)

    popt, pcov = curve_fit(power_law, x, y, p0=p0, sigma=yerr,
                            absolute_sigma=True, bounds=bounds, maxfev=20000)
    perr = np.sqrt(np.diag(pcov))
    return popt, perr


# ===========================================================================
# 6. Driver + plotting (ported from the main loop + canvases c3/c4)
# ===========================================================================
PLANE_COLORS = {0: "tab:blue", 1: "tab:orange", 2: "tab:green"}


def analyze(hit_dfs, particle="muon", dedx_col="dedx",
            rr_max_by_particle=None, theoretical_mpv_func=None,
            out_prefix="langau_rr", make_summary_plots=True, verbose=True):
    """
    hit_dfs: list of 3 per-plane dataframes (hit_dfs[0]=plane0, hit_dfs[1]=
             plane1, hit_dfs[2]=plane2), each with a 'tpc' column (0 or 1).

    Returns (all_results, fit_params):
      all_results[(plane, tpc)] -> per-rr-bin DataFrame of fit outputs
      fit_params[(plane, tpc)]  -> (popt, perr) of the power-law fit, or
                                    (None, None) if there weren't enough
                                    usable slices
    """
    if rr_max_by_particle is None:
        rr_max_by_particle = {"muon": 80.0, "pion": 40.0, "proton": 60.0}
    rr_max = rr_max_by_particle.get(particle, 80.0)

    all_results = {}
    fit_params = {}

    for plane, df in enumerate(hit_dfs):
        for tpc in (0, 1):
            if verbose:
                print(f"Plane {plane}, TPC {tpc}")
            sub = df[df["tpc"] == tpc]
            res = fit_rr_slices(sub, plane, tpc, dedx_col=dedx_col,
                                 rr_max=rr_max,
                                 theoretical_mpv_func=theoretical_mpv_func,
                                 verbose=verbose)
            all_results[(plane, tpc)] = res

            popt = perr = None
            if len(res) >= 3:
                try:
                    popt, perr = fit_sigma_vs_mpv(res)
                except RuntimeError as exc:
                    if verbose:
                        print(f"  power-law fit failed: {exc}")
            fit_params[(plane, tpc)] = (popt, perr)

    non_empty = [r.assign(plane=p, tpc=t) for (p, t), r in all_results.items() if len(r)]
    combined = pd.concat(non_empty, ignore_index=True) if non_empty else pd.DataFrame()

    if len(combined):
        combined.to_hdf(f"{out_prefix}_slices.h5", key="fits", mode="w",
                         format="table", complib="blosc", complevel=9)

    if make_summary_plots:
        plot_all_planes(all_results, fit_params, particle, out_prefix)
        plot_by_plane(all_results, fit_params, particle, out_prefix)

    return all_results, fit_params


def plot_all_planes(all_results, fit_params, particle, out_prefix):
    """Ported from canvas c3: all 6 (plane, tpc) power-law curves overlaid."""
    fig, ax = plt.subplots(figsize=(7, 6))
    for (plane, tpc), res in all_results.items():
        popt, _ = fit_params[(plane, tpc)]
        if popt is None or not len(res):
            continue
        color = PLANE_COLORS[plane]
        style = "-" if tpc == 0 else "--"
        xs = np.linspace(res["mpv_x"].min(), res["mpv_x"].max(), 200)
        side = "x < 0" if tpc == 0 else "x > 0"
        label = f"{side}, plane {plane}: {popt[0]:.2f} + {popt[1]:.2f}*x^{popt[2]:.2f}"
        ax.plot(xs, power_law(xs, *popt), style, color=color, label=label)

    ax.set_xlabel("MPV [MeV/cm]")
    ax.set_ylabel(r"$\sigma_G$ [MeV/cm]")
    title = {"muon": r"$\mu$ gaussian smearing", "pion": r"$\pi$ gaussian smearing",
             "proton": "proton gaussian smearing"}.get(particle, f"{particle} gaussian smearing")
    ax.set_title(title)
    ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(f"{out_prefix}_all_planes.pdf")
    plt.show()


def plot_by_plane(all_results, fit_params, particle, out_prefix):
    """Ported from canvas c4: one subplot per plane, tpc0 vs tpc1 overlaid."""
    fig, axes = plt.subplots(1, 3, figsize=(15, 5))
    for plane in range(3):
        ax = axes[plane]
        for tpc, color, marker in ((0, "tab:red", "o"), (1, "tab:blue", "s")):
            res = all_results.get((plane, tpc))
            popt, _ = fit_params.get((plane, tpc), (None, None))
            if res is None or not len(res):
                continue
            side = "x < 0" if tpc == 0 else "x > 0"
            ax.errorbar(res["mpv_x"], res["gsigma"], yerr=res["gsigma_err"],
                        fmt=marker, color=color, ms=4, label=f"{side} data")
            if popt is not None:
                xs = np.linspace(res["mpv_x"].min(), res["mpv_x"].max(), 200)
                ax.plot(xs, power_law(xs, *popt), "-", color=color,
                        label=f"{side}: {popt[0]:.2f} + {popt[1]:.2f}*x^{popt[2]:.2f}")
        ax.set_title(f"Plane {plane}")
        ax.set_xlabel("MPV [MeV/cm]")
        ax.set_ylabel(r"$\sigma_G$ [MeV/cm]")
        ax.legend(fontsize=7)
    fig.tight_layout()
    fig.savefig(f"{out_prefix}_by_plane.pdf")
    plt.show()



In [ ]:
pdg = 13
particle="muon"
if "pion" in bnb_path:
    pdg = 211
    particle = "pion"
    
theoretical_mpv = make_theoretical_mpv_func(hfit, pdg=pdg) 
all_results, fit_params = analyze(hit_dfs, particle=particle,
                                  theoretical_mpv_func=theoretical_mpv)

In [ ]:
fit_params

In [ ]:
def plot_slice_diagnostic(
    df,
    plane,
    tpc,
    rr_min,
    rr_max,
    hfit=None,
    pdg=13,
    mass=None,
    fit_params=None,
    dedx_col="dedx",
    nbins=100,
    hist_xmin=0.0,
    hist_xmax=10.0,
    first_stage_range=(0.0, 10.0),
):
    """Sanity-check plot for a single (plane, tpc, rr-range) slice: the raw dE/dx

    histogram, the two-stage Langau fit from section 3, and (if `hfit` is given)
    the theoretical PDF from `build_theoretical_pdf`, all on one axes. Useful for
    eyeballing a handful of slices before running the full `analyze()` loop
    over every rr bin.

    If `fit_params` is also given (the dict returned by `analyze()`, i.e.
    `fit_params[(plane, tpc)] = (popt, perr)` for the sigma_G(MPV) power
    law from `fit_sigma_vs_mpv`), a 4th curve is added: the theoretical
    PDF convolved with the Gaussian-smearing sigma that power law
    *predicts* at this slice's theoretical MPV. That's the actual
    end-to-end prediction of the calibration method (Bethe-Bloch/Landau
    theory + the resolution model fit from every other rr slice in this
    plane/tpc) -- it should land closest to the data if the whole method
    is self-consistent, unlike the raw (unconvolved) theory curve, which
    ignores detector resolution entirely. Skipped if `fit_params` has no
    entry for this (plane, tpc) (e.g. too few good slices to fit there).

    The Langau-fit curve and both theory curves are each rescaled so
    their peak matches the data histogram's peak -- their "natural"
    normalizations differ (the fit is in raw counts, the theory curves
    are unit-area PDFs times an approximate norm factor), so without this
    they can come out much taller/shorter than the data even when the
    underlying shape agrees. Rescaling to a common peak height makes this
    a pure shape comparison, and the shared height also means one y-limit
    covers everything.
    """
    sl = df[(df["tpc"] == tpc) & (df["rr"] >= rr_min) & (df["rr"] < rr_max) & (df["pitch"] <= 2)]
    rr_center = 0.5 * (rr_min + rr_max)

    fig, ax = plt.subplots(figsize=(8, 5.5))
    counts, edges, _ = ax.hist(
        sl[dedx_col].dropna(),
        bins=nbins,
        range=(hist_xmin, hist_xmax),
        histtype="stepfilled",
        alpha=0.3,
        color="steelblue",
        edgecolor="navy",
        label=f"data (N={len(sl)})",
    )
    bin_width = (hist_xmax - hist_xmin) / nbins
    x_eval = np.linspace(hist_xmin, hist_xmax, 400)
    data_max = counts.max() if len(counts) else 0.0
    curve_max = data_max  # track the overall peak across everything plotted

    hist = th1_from_series(
        sl[dedx_col],
        f"diag_p{plane}_t{tpc}",
        "",
        nbins,
        hist_xmin,
        hist_xmax,
    )
    fit = langau_fit_two_stage(hist, first_stage_range=first_stage_range)
    if fit is not None:
        width, mpv_reco, area, gsigma = fit["pars"]
        f_curve = ROOT.TF1(
            f"diag_curve_p{plane}_t{tpc}",
            ROOT.langau_cpp,
            hist_xmin,
            hist_xmax,
            4,
        )
        f_curve.SetParameters(width, mpv_reco, area, gsigma)
        y_fit = np.array([f_curve.Eval(xv) for xv in x_eval])
        if y_fit.max() > 0 and data_max > 0:
            y_fit = y_fit / y_fit.max() * data_max
        ax.plot(
            x_eval,
            y_fit,
            "g--",
            lw=2,
            label=f"Langau fit (MPV={mpv_reco:.2f})",
        )
        curve_max = max(curve_max, y_fit.max())

    if hfit is not None:
        mean_pitch = sl["pitch"].median() if "pitch" in sl else 0.3
        print(mean_pitch)
        mean_pitch = 0.32
        pdf = build_theoretical_pdf(
            hfit, pdg, rr_center, mean_pitch, mass=mass
        )
        theo_mpv = pdf.GetMaximumX()
        norm = len(sl) * bin_width
        y_theo = np.array([pdf.Eval(xv) * norm for xv in x_eval])
        if y_theo.max() > 0 and data_max > 0:
            y_theo = y_theo / y_theo.max() * data_max
        ax.plot(
            x_eval,
            y_theo,
            "r:",
            lw=2.5,
            label=f"theory, no smearing (MPV={theo_mpv:.2f})",
        )
        curve_max = max(curve_max, y_theo.max())

        popt = None
        if fit_params is not None:
            popt, _ = fit_params.get((plane, tpc), (None, None))
        if popt is not None:
            predicted_sigma_g = power_law(theo_mpv, *popt)
            y_pred = convolve_pdf_with_gaussian(pdf, predicted_sigma_g, x_eval)
            if y_pred.max() > 0 and data_max > 0:
                y_pred = y_pred / y_pred.max() * data_max
            # 🎨 Changed color to orange
            ax.plot(
                x_eval,
                y_pred,
                color="darkorange",
                linestyle="-",
                lw=2.5,
                label=rf"theory $\otimes$ predicted resolution ($\sigma_G$={predicted_sigma_g:.3f})",
            )
            curve_max = max(curve_max, y_pred.max())

    ax.set_ylim(0, curve_max * 1.15 if curve_max > 0 else 1.0)
    ax.set_xlabel(r"$dE/dx$ [MeV/cm]")
    ax.set_ylabel(f"hits / {bin_width:.2f} MeV/cm")
    ax.set_title(f"plane {plane}, tpc {tpc}, {rr_min:g} <= rr < {rr_max:g} cm")
    ax.legend(fontsize=9)
    fig.tight_layout()
    plt.show()
    return fig

import numpy as np

import numpy as np


import numpy as np

def convolve_pdf_with_gaussian(
    pdf, sigma_g, x_eval, n_sigma=5.0, grid_step=None
):
    """Numerically convolves `pdf` with a Gaussian of width `sigma_g` without shifting the MPV."""
    if sigma_g is None or not np.isfinite(sigma_g) or sigma_g <= 0:
        return np.array([pdf.Eval(x) for x in x_eval])

    x_eval = np.asarray(x_eval, dtype=float)

    # 1. Fine grid step (at least 20-30 points per sigma for low discretization bias)
    if grid_step is None:
        grid_step = sigma_g / 25.0

    # 2. Symmetric padding based on exact integer multiples of grid_step
    # to keep 0 perfectly centered in kernel space
    n_pad = int(np.ceil((n_sigma * sigma_g) / grid_step))
    pad_width = n_pad * grid_step

    # Extend evaluation range symmetrically
    grid_min = x_eval.min() - pad_width
    grid_max = x_eval.max() + pad_width
    grid = np.arange(grid_min, grid_max + grid_step, grid_step)

    # Evaluate base PDF
    pdf_vals = np.array([pdf.Eval(t) for t in grid])

    # 3. Create strictly symmetric odd-length kernel centered at 0
    k = np.arange(-n_pad, n_pad + 1)  # Length is strictly 2 * n_pad + 1 (odd)
    kernel_x = k * grid_step
    kernel = np.exp(-0.5 * (kernel_x / sigma_g) ** 2)
    kernel /= kernel.sum()  # Normalize discrete kernel area

    # 4. Perform convolution
    conv_vals = np.convolve(pdf_vals, kernel, mode="same")

    # 5. Interpolate back to original evaluation points
    return np.interp(x_eval, grid, conv_vals)


In [ ]:
print(df["pitch"].mean())

In [ ]:
hit_dfs = [mc_bnb_hit0_df, mc_bnb_hit1_df, mc_bnb_hit2_df]
rr_ranges = [(1, 2), (2, 3), (4, 5), (6, 7), (15, 16)]

for rr_min, rr_max in rr_ranges:
    plot_slice_diagnostic(df, 2, 0, rr_min, rr_max, hfit=hfit, pdg=pdg, fit_params=fit_params, dedx_col="dedx",
                           nbins=100, hist_xmin=0.0, hist_xmax=10.0)

In [ ]:
import matplotlib.pyplot as plt

# Filter pitch between 0 and 4
filtered_pitch = df[(df["pitch"] >= 0) & (df["pitch"] <= 2)]["pitch"].dropna()

plt.figure(figsize=(8, 5))
plt.hist(
    filtered_pitch,
    bins=50,
    range=(0, 4),  # Forces bin boundaries strictly within [0, 4]
    color="#1f77b4",
    edgecolor="black",
    alpha=0.7,
)

plt.xlabel("Pitch [cm]", fontsize=12)
plt.ylabel("Counts", fontsize=12)
plt.title("Distribution of Hit Pitch Values", fontsize=14)
plt.xlim(0, 2)  # Sets axis limits strictly from 0 to 4
plt.grid(axis="y", linestyle="--", alpha=0.5)

plt.tight_layout()
plt.show()